# Лабораторна робота № 1 — засоби моделювання Python

Варіант 3. Вовкотруб Олександр Віталійович, група ФБ-61мн.
КПІ ім. Ігоря Сікорського, ННФТІ, кафедра інформаційної безпеки.

Фігури Лісажу, поліном Чебишева, годограф Михайлова та модель банкомата.
Зведення обчислених величин наприкінці вивантажується у `data/results-summary.csv`.

## 0. Підготовка середовища

Клітинка нижче створює теки `assets/` і `data/` поруч із ноутбуком і, якщо
якоїсь бібліотеки бракує, доставляє її. Завдяки цьому ноутбук виконується
як локально, так і в Colab без жодних ручних кроків.

In [1]:
import importlib.util
import subprocess
import sys
from pathlib import Path


def _in_colab() -> bool:
    if "google.colab" in sys.modules:
        return True
    try:
        return importlib.util.find_spec("google.colab") is not None
    except (ImportError, ValueError):
        # find_spec імпортує батьківський пакет; без нього це не Colab
        return False


IN_COLAB = _in_colab()

REQUIRED = ['numpy', 'matplotlib']
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

ASSETS = Path("assets")
DATA = Path("data")
ASSETS.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

In [2]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Liberation Serif", "DejaVu Serif"],
    "font.size": 11,
    "figure.dpi": 150,
    "savefig.bbox": "tight",
    "axes.grid": True,
    "grid.alpha": 0.3,
})

RESULTS: list[dict] = []


def record(section: str, name: str, value, note: str = "") -> None:
    """Запам'ятати число для `data/results-summary.csv` і показати його."""
    RESULTS.append({"розділ": section, "показник": name, "значення": value, "примітка": note})
    print(f"{section} | {name} = {value}  {note}")

## 1. Фігури Лісажу

Параметрична крива $x = \cos(\omega_1 t + \varphi_1)$, $y = \cos(\omega_2 t + \varphi_2)$.
Досліджуємо, як змінюється зображення при зміні різниці фаз
$\Delta\varphi = \varphi_1 - \varphi_2$.

In [3]:
def lissajous(w1: float, w2: float, dphi: float, t: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Координати фігури Лісажу з різницею фаз dphi."""
    return np.cos(w1 * t + dphi), np.cos(w2 * t)


def phase_grid(ratio: float, fname: str, title: str, periods: float = 1.0) -> None:
    """Сітка фігур для Δφ = 0, π/6, π/4, π/3, π/2, π при заданому ω1/ω2."""
    phases = [(0, "0"), (np.pi / 6, r"$\pi/6$"), (np.pi / 4, r"$\pi/4$"),
              (np.pi / 3, r"$\pi/3$"), (np.pi / 2, r"$\pi/2$"), (np.pi, r"$\pi$")]
    t = np.linspace(0, 2 * np.pi * periods, 4000)
    fig, axes = plt.subplots(2, 3, figsize=(7.2, 4.8))
    for ax, (dphi, label) in zip(axes.ravel(), phases):
        x, y = lissajous(ratio, 1.0, dphi, t)
        ax.plot(x, y, lw=1.2, color="#1f3b73")
        ax.set_title(rf"$\Delta\varphi = ${label}", fontsize=10)
        ax.set_aspect("equal")
        ax.set_xticks([-1, 0, 1]); ax.set_yticks([-1, 0, 1])
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(ASSETS / fname)
    plt.close(fig)


phase_grid(1.0, "01-lissajous-rank1.png",
           r"Фігури Лісажу 1-го рангу ($\omega_1/\omega_2 = 1$)")
phase_grid(2.0, "02-lissajous-rank2.png",
           r"Фігури Лісажу 2-го рангу ($\omega_1/\omega_2 = 2$)", periods=1.0)

### Раціональне та ірраціональне відношення частот

При раціональному $\omega_1/\omega_2 = m/n$ крива **замкнена** і період
скінченний. При ірраціональному вона не замикається ніколи і з часом
щільно заповнює квадрат $[-1,1]^2$ — це відповідь на контрольне питання 1.5.1.

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.6))
for ax, (w1, label, tmax) in zip(axes, [
    (1.5, r"$\omega_1/\omega_2 = 3/2$ (раціональне, замкнена)", 4 * np.pi),
    (np.sqrt(2), r"$\sqrt{2}$, $t \leq 20\pi$", 20 * np.pi),
    (np.sqrt(2), r"$\sqrt{2}$, $t \leq 400\pi$", 400 * np.pi),
]):
    t = np.linspace(0, tmax, 60000)
    x, y = lissajous(w1, 1.0, np.pi / 2, t)
    ax.plot(x, y, lw=0.4, color="#1f3b73")
    ax.set_title(label, fontsize=9)
    ax.set_aspect("equal")
    ax.set_xticks([-1, 0, 1]); ax.set_yticks([-1, 0, 1])
fig.tight_layout()
fig.savefig(ASSETS / "03-lissajous-irrational.png")
plt.close(fig)

# Кількість дотиків до сторін прямокутника дає відношення частот:
# для ω1/ω2 = m/n фігура торкається горизонтальної сторони m разів,
# вертикальної — n разів.
record("Лісажу", "відношення частот, раціональний приклад", "3/2",
       "замкнена крива, 3 дотики по вертикалі та 2 по горизонталі")
record("Лісажу", "відношення частот, ірраціональний приклад", "sqrt(2)",
       "крива не замикається, заповнює квадрат")

Лісажу | відношення частот, раціональний приклад = 3/2  замкнена крива, 3 дотики по вертикалі та 2 по горизонталі
Лісажу | відношення частот, ірраціональний приклад = sqrt(2)  крива не замикається, заповнює квадрат


## 2. Поліном Чебишева

$T_n(x) = \cos(n \arccos x)$ на $[-1, 1]$. Обраний порядок $n = 5$.
Тотожність перевіряємо трьома незалежними способами: означенням через косинус,
рекурентною формулою $T_{n+1} = 2x T_n - T_{n-1}$ і `numpy.polynomial.chebyshev`.

In [5]:
N_CHEB = 5


def chebyshev(n: int, x: np.ndarray) -> np.ndarray:
    """T_n(x) за означенням через косинус; |x| <= 1."""
    return np.cos(n * np.arccos(x))


def chebyshev_recurrent(n: int, x: np.ndarray) -> np.ndarray:
    """T_n(x) за рекурентною формулою."""
    t_prev, t_cur = np.ones_like(x), x
    if n == 0:
        return t_prev
    for _ in range(n - 1):
        t_prev, t_cur = t_cur, 2 * x * t_cur - t_prev
    return t_cur


x = np.linspace(-1, 1, 2001)
t_def = chebyshev(N_CHEB, x)
t_rec = chebyshev_recurrent(N_CHEB, x)
t_np = np.polynomial.chebyshev.Chebyshev.basis(N_CHEB)(x)

err_rec = float(np.abs(t_def - t_rec).max())
err_np = float(np.abs(t_def - t_np).max())
assert err_rec < 1e-12, err_rec
assert err_np < 1e-12, err_np
record("Чебишев", "порядок n", N_CHEB)
record("Чебишев", "макс. розбіжність означення vs рекурентна", f"{err_rec:.2e}")
record("Чебишев", "макс. розбіжність означення vs numpy", f"{err_np:.2e}")

roots = np.cos((2 * np.arange(N_CHEB) + 1) * np.pi / (2 * N_CHEB))
record("Чебишев", "кількість нулів на [-1,1]", len(roots))
record("Чебишев", "кількість екстремумів ±1", N_CHEB + 1)

fig, ax = plt.subplots(figsize=(7, 3.4))
for n, style in [(3, ":"), (4, "--"), (N_CHEB, "-")]:
    ax.plot(x, chebyshev(n, x), style, lw=1.6 if n == N_CHEB else 1.0,
            label=rf"$T_{{{n}}}(x)$")
ax.plot(roots, np.zeros_like(roots), "o", ms=5, color="crimson",
        label=rf"нулі $T_{{{N_CHEB}}}$")
ax.axhline(1, color="gray", lw=0.6); ax.axhline(-1, color="gray", lw=0.6)
ax.set_xlabel("$x$"); ax.set_ylabel("$T_n(x)$")
ax.set_title("Поліноми Чебишева першого роду")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig(ASSETS / "04-chebyshev-t5.png")
plt.close(fig)

Чебишев | порядок n = 5  
Чебишев | макс. розбіжність означення vs рекурентна = 1.72e-15  
Чебишев | макс. розбіжність означення vs numpy = 2.55e-15  
Чебишев | кількість нулів на [-1,1] = 5  
Чебишев | кількість екстремумів ±1 = 6  


## 3. Годограф Михайлова — варіант 3

$$D(p) = p^{10} + 7p^{8} + 11p^{7} + 5p + 1$$

**Необхідна умова стійкості** — усі коефіцієнти характеристичного полінома
додатні. Тут $a_9 = a_6 = a_5 = a_4 = a_3 = a_2 = 0$, тож вона **не виконується**,
і система нестійка ще до побудови годографа.

Критерій Михайлова: для стійкості потрібно, щоб приріст аргументу
$\arg D(i\omega)$ при $\omega: 0 \to +\infty$ дорівнював $n\pi/2$, тобто щоб
годограф послідовно обійшов $n$ квадрантів проти годинникової стрілки.
За принципом аргументу для полінома степеня $n$ з $k$ коренями у правій
півплощині приріст дорівнює $(n - 2k)\pi/2$ — тож виміряний приріст
**дає число правих коренів** незалежно від прямого їх обчислення.

In [6]:
D_VAR3 = [1, 0, 7, 11, 0, 0, 0, 0, 0, 5, 1]   # p^10 + 7p^8 + 11p^7 + 5p + 1
D_STABLE = [1, 6, 11, 6]                       # (p+1)(p+2)(p+3)


def mikhailov_increment(coeffs, w_max: float = 4000.0, n_points: int = 2_000_001) -> float:
    """Приріст arg D(iω) на [0, w_max], рад."""
    w = np.linspace(0.0, w_max, n_points)
    phase = np.unwrap(np.angle(np.polyval(coeffs, 1j * w)))
    return float(phase[-1] - phase[0])


def plot_mikhailov(coeffs, fname: str, title: str, w_max: float, w_zoom: float) -> None:
    """Годограф у двох масштабах: загальний вигляд і околиця початку координат."""
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
    for ax, wm, sub in zip(axes, (w_max, w_zoom),
                           ("загальний вигляд", "околиця початку координат")):
        w = np.linspace(0.0, wm, 400000)
        d = np.polyval(coeffs, 1j * w)
        ax.plot(d.real, d.imag, lw=1.0, color="#1f3b73")
        ax.plot([d.real[0]], [d.imag[0]], "o", ms=5, color="crimson")
        ax.axhline(0, color="gray", lw=0.6); ax.axvline(0, color="gray", lw=0.6)
        ax.set_xlabel(r"$X = \mathrm{Re}\,D(i\omega)$")
        ax.set_ylabel(r"$Y = \mathrm{Im}\,D(i\omega)$")
        ax.set_title(sub, fontsize=9)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(ASSETS / fname)
    plt.close(fig)


# --- варіант 3 ---
n3 = len(D_VAR3) - 1
positive_coeffs = all(c > 0 for c in D_VAR3)
roots3 = np.roots(D_VAR3)
k3 = int((roots3.real > 0).sum())
inc3 = mikhailov_increment(D_VAR3)

record("Михайлов", "степінь n", n3)
record("Михайлов", "усі коефіцієнти додатні", positive_coeffs,
       "необхідна умова стійкості не виконується")
record("Михайлов", "коренів з Re>0", f"{k3} з {n3}")
record("Михайлов", "max Re кореня", f"{roots3.real.max():.4f}")
record("Михайлов", "приріст arg D(iw), pi", f"{inc3 / np.pi:.4f}",
       f"для стійкості потрібно {n3 / 2:.1f} pi")
record("Михайлов", "k за принципом аргументу", int(round((n3 - 2 * inc3 / np.pi) / 2)),
       "збігається з прямим підрахунком коренів")

assert not positive_coeffs
assert k3 == 6
assert abs(inc3 / np.pi - (-1.0)) < 0.01, inc3 / np.pi
assert int(round((n3 - 2 * inc3 / np.pi) / 2)) == k3

plot_mikhailov(D_VAR3, "05-mikhailov-var3.png",
               r"Годограф Михайлова: $D(p) = p^{10} + 7p^{8} + 11p^{7} + 5p + 1$",
               w_max=2.0, w_zoom=0.4)

Михайлов | степінь n = 10  
Михайлов | усі коефіцієнти додатні = False  необхідна умова стійкості не виконується
Михайлов | коренів з Re>0 = 6 з 10  
Михайлов | max Re кореня = 0.7514  
Михайлов | приріст arg D(iw), pi = -1.0000  для стійкості потрібно 5.0 pi
Михайлов | k за принципом аргументу = 6  збігається з прямим підрахунком коренів


### Контрольне питання 1.5.2 — приклад стійкої системи 3-го порядку

$$D(p) = p^{3} + 6p^{2} + 11p + 6 = (p+1)(p+2)(p+3)$$

Усі корені дійсні від'ємні; годограф послідовно проходить три квадранти,
приріст аргументу дорівнює $3\pi/2$ — система стійка за Михайловим.

In [7]:
n_st = len(D_STABLE) - 1
roots_st = np.roots(D_STABLE)
inc_st = mikhailov_increment(D_STABLE)

record("Михайлов (стійкий приклад)", "корені", np.round(np.sort(roots_st.real), 4).tolist())
record("Михайлов (стійкий приклад)", "приріст arg D(iw), pi", f"{inc_st / np.pi:.4f}",
       f"потрібно {n_st / 2:.1f} pi")

assert (roots_st.real < 0).all()
assert abs(inc_st / np.pi - n_st / 2) < 0.01

plot_mikhailov(D_STABLE, "06-mikhailov-stable-example.png",
               r"Годограф Михайлова: $D(p) = p^{3} + 6p^{2} + 11p + 6$ — стійка",
               w_max=20.0, w_zoom=4.0)

Михайлов (стійкий приклад) | корені = [-3.0, -2.0, -1.0]  
Михайлов (стійкий приклад) | приріст arg D(iw), pi = 1.4995  потрібно 1.5 pi


## 4. Банкомат

Скінченний автомат за схемою Рис.1 умови. Реалізація — у `atm.py`
Повний код автомата наведено нижче окремою клітинкою.

Ноутбук виконується неінтерактивно, тому введення подається **сценарієм** —
ітератором рядків. Інтерактивний режим відрізняється лише джерелом ітератора.

In [8]:
"""Модель банкомата — скінченний автомат за схемою Рис.1 умови ЛР № 1.

Стани a0…a13 та переходи між ними відтворені буквально за схемою.
Гроші зберігаються в копійках цілим числом, щоб не було похибок float.
Введення подається ітератором рядків: у ноутбуці це заготовлений сценарій,
в інтерактивному режимі — обгортка над input().
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Iterator

NOMINALS = (5000, 1000, 500, 200, 100, 50, 20, 10)  # гривні


def banknotes(amount_kop: int) -> dict[int, int]:
    """Розкласти суму на банкноти (жадібно). Копійки видати неможливо."""
    if amount_kop <= 0:
        raise ValueError("сума має бути додатною")
    if amount_kop % 100:
        raise ValueError("банкомат видає лише цілі гривні")
    left, out = amount_kop // 100, {}
    for nominal in NOMINALS:
        count, left = divmod(left, nominal)
        if count:
            out[nominal] = count
    if left:
        raise ValueError(f"сума не розкладається на банкноти, лишок {left} грн")
    return out


def hryvnia(amount_kop: int) -> str:
    return f"{amount_kop // 100}.{amount_kop % 100:02d}"


@dataclass
class ATMLog:
    states: list[str] = field(default_factory=list)
    output: list[str] = field(default_factory=list)
    balance_kop: int = 0
    card_blocked: bool = False


class ATM:
    def __init__(self, balance_kop: int, allowed_types: set[int],
                 allowed_codes: set[int], max_attempts: int = 2) -> None:
        self.balance_kop = balance_kop
        self.allowed_types = allowed_types
        self.allowed_codes = allowed_codes
        self.max_attempts = max_attempts

    def run(self, inputs: Iterator[str]) -> ATMLog:
        log = ATMLog(balance_kop=self.balance_kop)

        def say(text: str) -> None:
            log.output.append(text)

        def enter(state: str) -> None:
            log.states.append(state)

        enter("a0"); say("Вставте картку")
        enter("a1")
        card_type = int(next(inputs))
        if card_type not in self.allowed_types:
            enter("a13"); say("Тип картки не підтримується цим банкоматом")
            return log

        enter("a2"); say("Введіть код")
        enter("a3")
        code = int(next(inputs))
        attempt = 1
        while code not in self.allowed_codes and attempt < self.max_attempts:
            enter("a11"); say("Невірний код. Введіть ще раз")
            code = int(next(inputs))
            attempt += 1
        if code not in self.allowed_codes:
            enter("a12"); say("Невірний код. Картку заблоковано")
            log.card_blocked = True
            return log

        while True:
            enter("a4")
            say("Меню: 1 — стан рахунку, 2 — зняття грошей, 3 — вихід")
            choice = next(inputs).strip()

            if choice == "1":
                enter("a5"); say(f"Стан рахунку: {hryvnia(log.balance_kop)} грн")
                continue

            if choice == "2":
                enter("a6"); say("Введіть суму")
                amount_kop = round(float(next(inputs)) * 100)
                enter("a7")
                try:
                    bills = banknotes(amount_kop)
                except ValueError as exc:
                    say(f"Неможливо видати: {exc}. Доступні банкноти: {NOMINALS}")
                    continue
                if amount_kop > log.balance_kop:
                    enter("a9"); say("Недостатньо коштів на рахунку")
                    continue
                enter("a8")
                log.balance_kop -= amount_kop
                say("Видано: " + ", ".join(f"{n} грн x{c}" for n, c in bills.items()))
                say(f"Залишок: {hryvnia(log.balance_kop)} грн")
                continue

            enter("a10"); say("Дякуємо, до побачення")
            return log

In [9]:
import random

random.seed(3)   # номер варіанта — щоб прогін відтворювався

ALLOWED_TYPES = {1, 2}
ALLOWED_CODES = {1234, 4321}
START_KOP = random.randint(0, 20_000) * 100 + random.randint(0, 99)

record("Банкомат", "початковий рахунок, грн", hryvnia(START_KOP),
       "цілі гривні та цілі копійки, ініціалізація випадкова")
record("Банкомат", "допустимі типи карток", sorted(ALLOWED_TYPES))
record("Банкомат", "спроб уведення коду", 2)

SCENARIOS = {
    "успішне зняття": ["1", "1234", "2", "300", "3"],
    "перегляд стану та вихід": ["1", "1234", "1", "3"],
    "недостатньо коштів": ["2", "4321", "2", "50000", "3"],
    "невірний код двічі": ["1", "1111", "2222"],
    "непідтримуваний тип картки": ["9"],
    "спроба зняти копійки": ["1", "1234", "2", "100.50", "3"],
}

for name, script in SCENARIOS.items():
    machine = ATM(START_KOP, ALLOWED_TYPES, ALLOWED_CODES)
    log = machine.run(iter(script))
    print(f"\n=== {name} ===")
    print("шлях:", " -> ".join(log.states))
    for line in log.output:
        print("   ", line)
    record("Банкомат", f"шлях: {name}", " -> ".join(log.states))

Банкомат | початковий рахунок, грн = 7797.75  цілі гривні та цілі копійки, ініціалізація випадкова
Банкомат | допустимі типи карток = [1, 2]  
Банкомат | спроб уведення коду = 2  

=== успішне зняття ===
шлях: a0 -> a1 -> a2 -> a3 -> a4 -> a6 -> a7 -> a8 -> a4 -> a10
    Вставте картку
    Введіть код
    Меню: 1 — стан рахунку, 2 — зняття грошей, 3 — вихід
    Введіть суму
    Видано: 200 грн x1, 100 грн x1
    Залишок: 7497.75 грн
    Меню: 1 — стан рахунку, 2 — зняття грошей, 3 — вихід
    Дякуємо, до побачення
Банкомат | шлях: успішне зняття = a0 -> a1 -> a2 -> a3 -> a4 -> a6 -> a7 -> a8 -> a4 -> a10  

=== перегляд стану та вихід ===
шлях: a0 -> a1 -> a2 -> a3 -> a4 -> a5 -> a4 -> a10
    Вставте картку
    Введіть код
    Меню: 1 — стан рахунку, 2 — зняття грошей, 3 — вихід
    Стан рахунку: 7797.75 грн
    Меню: 1 — стан рахунку, 2 — зняття грошей, 3 — вихід
    Дякуємо, до побачення
Банкомат | шлях: перегляд стану та вихід = a0 -> a1 -> a2 -> a3 -> a4 -> a5 -> a4 -> a10  

=== 

### Контрольне питання 1.5.3

*Чи містить Python засоби, щоб задати повторний ввід коду без стану `a11`?*

Так. Стан `a11` існує у схемі тому, що це автомат, де **кожна дія — окрема
вершина**: «запросити ввести код ще раз» — така сама дія, як і решта.
У Python цикл робить окрему вершину зайвою: лічильник спроб і `for`/`while`
навколо `a3` дають ту саму поведінку без `a11`.

In [10]:
def check_code_without_a11(codes, allowed, max_attempts=2):
    """Той самий контроль коду, але циклом — без окремого стану a11."""
    states = ["a2"]
    for attempt, code in zip(range(1, max_attempts + 1), codes):
        states.append("a3")
        if code in allowed:
            return states + ["a4"], False
        if attempt == max_attempts:
            return states + ["a12"], True
    return states + ["a12"], True


path_loop, blocked_loop = check_code_without_a11([1111, 2222], ALLOWED_CODES)
path_fsm = ATM(START_KOP, ALLOWED_TYPES, ALLOWED_CODES).run(iter(["1", "1111", "2222"]))

record("Банкомат", "шлях без a11 (цикл)", " -> ".join(path_loop))
record("Банкомат", "шлях зі схеми (з a11)", " -> ".join(path_fsm.states))
assert blocked_loop and path_fsm.card_blocked
# Різниця лише у вершинах a0/a1 та наявності a11 — результат однаковий.

Банкомат | шлях без a11 (цикл) = a2 -> a3 -> a3 -> a12  
Банкомат | шлях зі схеми (з a11) = a0 -> a1 -> a2 -> a3 -> a11 -> a12  


## Зведення результатів

Усі числа звіту беруться з цього файла, а не переписуються руками.

In [11]:
import csv

with open(DATA / "results-summary.csv", "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=["розділ", "показник", "значення", "примітка"])
    writer.writeheader()
    writer.writerows(RESULTS)
print(f"записано {len(RESULTS)} рядків у {DATA / 'results-summary.csv'}")

записано 26 рядків у data/results-summary.csv
